# Build Your Own Call Recogniser

_Integrating Passive Acoustic Monitoring with AI for Scalable Biodiversity Tracking_

## Introduction

Welcome to the agile modelling Python notebook.

### What is a Python notebook?

A Python notebook allows you to run Python code in a Python environment. The Python notebook is running in a virtual machine in the cloud.

### Do I need to be familiar with Python?

No, you do not need to be familiar with Python to work through the notebook. You will just need to make some very small changes to parameter values.

### Notebook overview

In this notebook, we will use a process called "[agile modeling](https://arxiv.org/abs/2302.12948)" to build and incrementally improve a classifier for acoustic analysis, starting from a single classified example. The process uses embeddings provided by the [Perch model](https://www.kaggle.com/models/google/bird-vocalization-classifier). These are the steps we will take:

1. Setup
2. Configure the Perch agile modelling modules
3. Initialise a database of embeddings
4. Search for recordings similar to an example you provide
5. Build a machine learning classifier model from the search results
6. Search your recordings based on the results of the classifier
7. Improve your classifier further using these search results
8. Save your classifier for future use and use it for detection

The agile modelling process is described in more detail in [these slides](https://docs.google.com/presentation/d/e/2PACX-1vTfvoBvCi_V72s0RiIcmFNdnZDcPDCDl-omBbODJ3sz3_IxD5kd1zJjd-J8AR7PE_DgxO-FWDjyP7Mb/pub?start=false&loop=false&delayms=3000&slide=id.g2d63d0c2ccf_0_3915) and in the following diagram:

<div>
<img src="https://storage.googleapis.com/chirp-public-bucket/esa-2024/agile_modelling_workflow.png" width="800"/>
</div>

## 1. Setup

This notebook uses several python "packages".  These have been preinstalled into this environment, but to make use of them in this notebook we need to import them


In [ ]:
#@title Install the `perch` package and import requirements
#@markdown <font color='green'>← Run this cell to import the required packages.</font>
#@markdown
#@markdown


import os
from pathlib import Path

from recognizer_workshop.engine.agile_engine import agile_config, agile_state
from recognizer_workshop.ui.notebook.adapter import NotebookUI

from recognizer_workshop.agile_modeling_helpers import (
    list_audio_files,
    labels_from_event_taggings,
    add_labels_to_db,
)

from recognizer_workshop.engine.services.search import resolve_query_uri

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'


## 2. Configure the Perch agile modelling modules

Here we set some configuration for names and local filepaths and initialize our agile modeling workflow.

In [ ]:

# The purpose of the `annotator_id` variable below is to keep track of
# who provided each annotation when collaborating across multiple
# annotators. This is not strictly necessary for this workhop, but can't hurt.
annotator_id = "phil"  

db_name = None

# This is the location that this tutorial will use to save data. The "~" your home directory, which 
# you can navigate in the file browser on the left side of the screen.
data_folder = Path('~/data').expanduser()

config = agile_config(
    db_path=data_folder / 'db' / db_name, # path to the pre-generated SQLite database of embeddings
    annotator_id=annotator_id,
    search_dataset_name="search_set",
    embeddings_folder=None, 
    labeled_examples_folder=data_folder / 'labeled_examples',
    models_folder=data_folder / 'models',
    predictions_folder=data_folder / 'predictions',
)

Path(config.labeled_examples_folder).mkdir(exist_ok=True, parents=True)

config.baw_config['domain'] = 'api.ecosounds.org'

agile = agile_state(config)
ui = NotebookUI(annotator_id=config.annotator_id)

## Configure access to your Ecosounds project

We will be loading audio from [Ecosounds](https://www.ecosounds.org), an online repository of ecoacoustic recordings. If working with a
private Ecosounds project, we need to provide the Ecosounds *auth token* associated with your ecosounds account.

To find your ecosounds token, go to https://www.ecosounds.org/my_account and in the bottom left, click on the button to copy the token like so:

 <div><img src="https://storage.googleapis.com/chirp-public-bucket/esa-2024/ecosounds_token.png" width="800"/></div>

Because this is a secret, we should avoid saving it in plain text in the notebook, so we will set it up in an environment variable.

1. In the working directory create a file named `.env`
    - If you are not sure where your working directory is, run the cell below and it should create it for you. 
2. In this .env file, put the line `BAW_AUTH_TOKEN=abc123xyz` (replace `abc123xyz` with the token you copied from Ecosounds)

In [ ]:

# this should automatically prompt for the token and create the .env file
agile.init_workbench()

## 3. Initalize a database of embeddings

Run the next cell to initialise the database

In [ ]:

agile.initialize()

# Select the species that you are targeting

Run the next cell to select a label. This label will be applied to any search results. 

The label must match an existing tag on ecosounds. If your target species is not listed, let us know and we can add it. 


In [ ]:

target_species = 'Grey Range Thick-billed Grasswren'

call_types = ["call", "song", "seep"] # or None / []

ui.pick_label(agile, initial_species=target_species, initial_call_types=call_types)

In [ ]:
agile.label

### What is an embedding?

Think of an embedding as a summary of the audio waveform in the form of an array of numbers. In the context of deep learning (which Perch relies on), that array of numbers is obtained by passing a _spectrogram_ representation of the audio through a deep neural network.

<div>
<img src="https://storage.googleapis.com/chirp-public-bucket/esa-2024/what_is_an_embedding.png" width="800"/>
</div>

The exact details can be safely ignored for the purpose of this tutorial, but the neural network was constructed and trained in such a way that the relationship between the embeddings it outputs tend to be consistent with the semantic relationships between the audio waveforms (e.g., passing two different vocalizations for the same animal species through the neural network will produce similar embeddings).

We will exploit this property in our agile modelling workflow.

The first step in the workflow is therefore to compute an embedding with Perch for every 5-second audio clip extracted from our audio recordings. In the interest of time, those embeddings have already been computed and assembled into a database.


### Add labels to the database

We will grab all the annotations from the ecosounds recordings in your search set and add them to the database so they can be used as labeled examples for training. 

In [ ]:
region_id = 85

labels = labels_from_event_taggings(
    agile.baw_api,
    filter_params={"region_id": [region_id]},
    target_labels=agile.get_target_labels(),  # IMPORTANT
)
add_labels_to_db(agile, config, labels)

## 4. Search for recordings similar to the annotator-provided example

Here, we take a single example and find the examples in our search set which most closely match that example. This is a way to get started with a labelled training set.

You have a few options to provide examples.

### A. Upload some short clips

On the file explorer on the left, `config.labeled_examples_folder`, you can upload short wav files to use as search queries.
If you are unsure which folder that is, running this cell will tell you

In [ ]:

audio_files = list_audio_files(config.labeled_examples_folder)
audio_files

### B. Provide a URL or Xeno Canto ID

You can also directly provide a URL pointing to the example or a Xeno-Canto ID in the form `xc123456`.

## Load the query audio 

The `query` below can be
1. one of the integer indices listed after running the cell for option A above; or
2. a URL, filepath, or Xeno-Canto ID (in the form `xc123456`).

Running the cell will display the example and allow you to select the5-second portion of it to use.

**NOTE: If your example is too long, this can make the selection of the 5-second segment a bit more difficult.**


In [ ]:

query = 'XC979796'

uri = resolve_query_uri(query, config.labeled_examples_folder)

samples = ui.select_query_window(uri)

## Embed the query and retrieve most similar candidates

The next step is to generate an embedding for the 5-second example and then compare it against the embeddings in the database to find the most similar 5-second clips from your search dataset.

Parameters:

- The `num_results` parameter controls the number of search results to present. Larger numbers allow to annotate more search results at a but going through the results requires more annotator time.

- When leaving `target_score` to None, the clips being surfaced will be the top `num_results` most similar clips with respect to the provided query example. However if you find that all the results are positive results almost the same as your query, can however be useful to retrieve embeddings with different levels of similarity, for instance to get good "negative" training examples to contrast against the "positive" matches. 

Running this cell will display a histogram of scores like this one:

<img src="https://storage.googleapis.com/chirp-public-bucket/esa-2024/logits_distribution.png" width="300"/>

The x-axis represents some arbitrary similarity "score", and the y-axis
represents how many embeddings in the database share that score. In
the example above, a `target_score` of 0.0 would for instance retrieve
embeddings in the database whose similarity "score" is closest to 0.0
and which are therefore amongst the most dissimilar to the provided
example.

**NOTE: You can leave the options below unchanged when first running this cell.**

In [ ]:

num_results = 10  

target_score = None

agile.embed_query()

agile.search_with_query(
    num_results=num_results,
    # When working with really large datasets it may be necessary for
    # performance reasons to look at a smaller (random) subset of the database
    # entries to perform the search. Replacing None with an integer argument
    # would limit the search to a random subset of that size.
    sample_size=None,
    target_score=target_score,
)

We are now ready to look at our first search results. For each example you can look at a spectrogram and listen to the audio, then apply a positive label if it is a positive match or a negative label if it's not a match.

Click the label below each recording to annotate it: click once to turn it green (positive label), twice to turn it orange (negative label), or leave it unclicked (or click a third times to reset) if you don't want to apply any label to the recording (i.e., you don't want to add it to your labelled training set at all).

 **NOTE: loading the spectrograms can sometimes fail. If you see some examples that failed to load, try running the cell a second time before starting your labelling.**


In [ ]:

label_key = agile.get_target_labels()[0]
ui.show_results(agile, label_key)

### Save the annotations
Run this cell to save your annotations. This will save the newly labelled examples to the database.

In [ ]:

verified = ui.harvest_labels()
agile.save_labels(verified)

Repeat the above cycle with a few different audio queries:

1. Load the query audio.
2. Embed the query and retrieve most similar candidates.
3. Inspect and annotate the search results.
4. Save the annotations.

If your target species has multiple call types, it would be a good idea to search for at least one of each call type.

## 5. Build a machine learning classifier model from the search results

Now that we have labelled a number of our embedded audio clips in the search set, we have what we need to train and evaluate a classifier.

At a high level, you can think of our embeddings as points on a map. From that perspective, classifying audio clips as "positive" (match) or "negative" (not a match) can be thought of as figuring out "territories" on our map for positives and negatives. If an embedding for a particular audio clip falls into the "positives" territory, it is classified as a positive, and vice versa.

<div>
<img src="https://storage.googleapis.com/chirp-public-bucket/esa-2024/classification.png" width="800"/>
</div>

To continue with the analogy, our territories are defined using "capital cities": we place a capital city for positives and one for for negatives on our map, and the territory that any point on our map belongs to is determined by which capital city the point is closest to.

"Training" the classifier therefore reduces to finding the two points on our map at which to place the capital cities such that the territories they claim align with our labelled audio clips. In an ideal case, the capitals should be located such that all positively-labelled clips in our dataset are in the "positives" territory, and vice versa.

The process by which we figure out the locations of our capital cities is beyond the scope of this tutorial, and the knobs one needs to tune to control the behavior of that process can be safely ignored and left to their default settings below. Refer to the short explanation under each parameter for more details.

# Train the classifier

## The following are three parameters of the algorithm that trains the recognizer.

`learning_rate` How much to update the model weights at each step
`num_steps` How many times to look at all the labeled examples when training
`batch_size` how many labeled examples to look at at once

# The following are to do with the labelled data inputs


`train_ratio` A random subset of the labelled audio is not used to train the model, but instead is used to test the model. This is so we know roughly how well the model does on classifying examples that it has never seen before.


`weak_neg_weight` In your database we have a lot of audio, most of which is probably
not your target. By taking some random clips from your unlabelled
audio and treating them as negative examples, we can train on a wider
variety of negative examples than what has been explicitly labelled
as negative. However, because we don't know for sure that this process
didn't choose a positive example by chance, we give each one less
importance in the training.

`weak_negatives_batch_size`  How many of these randomly chosen examples to include for each batch (on top of the number in the strongly labelled batch).


In [ ]:


# How much to update the locations at each step.
learning_rate = 1e-3
# How many steps to do:
num_steps = 101
# How many labelled examples to use at each step to determine an update
# direction.
batch_size = 32

train_ratio = 0.9

weak_neg_weight = 0.05

weak_negatives_batch_size = 16

agile.train_classifier(learning_rate, weak_neg_weight, num_steps, train_ratio, batch_size, weak_negatives_batch_size)


## 6. Search your recordings based on the results of the classifier

Now that we have a trained classifier, we can follow the same process as for the single example query. We search the database for more examples using the classifier, label them, then re-train the classifier with the new examples.

The classifier outputs a score for each example in the search set. Large positive/negative values mean the classifier predicts a positive/negative label with strong confidence, whereas values around zero mean the classifier's confidence is low.

When searching using the classifier, we can look for examples with the highest score by setting `target_score` to `None`.  This might get us more positive examples but these probably won't improve the classifier much, because they already have a high score. More useful is to search for those examples that the classifier is least sure about, by setting `target_score` to `0`.  Try setting the target score to None, 0 and possibly some other values depending on how many positive examples come back from each of those.

Run the cell below to search the database using the classifier.

In [ ]:

num_results = 10  

target_score = None 

agile.search_with_classifier(
    num_results=num_results,
    # When working with really large datasets it may be necessary for
    # performance reasons to look at a smaller (random) subset of the database
    # entries to perform the search. Replacing None with an integer argument
    # would limit the search to a random subset of that size.
    sample_size=None,
    target_score=target_score,
)



### Inspect and annotate the search results

Run this cell to inspect and annotate the retrieved recordings in the database.

In [ ]:

for label_key in agile.classifier.classes:
    print(f"=== Results for: {label_key} ===")
    ui.show_results(agile, label_key)

# Save the annotations

After you have manually labelled the search results, run this cell to save your annotations.

In [ ]:

verified = ui.harvest_labels()
agile.save_labels(verified)

## 7. Improve your classifier further using these search results

You can now go back to the _Build a machine learning classifier model_ section to retrain the classifier based on all annotations provided so far.

## Saving your classifier and running inference

The trained classifier consists of the following elements
1. The *weights* of the model (how to multiply and add the embedding values together to produce higher scores for the examples of the target class than other examples), which were learned during training.
2. The model *bias* (how to shift the scores so that scores for positive examples are positive and vice-versa), also learned during training.
3. The list of labels (class names) corresponding to the output scores
4. Some metadata related to the model that created the embeddings, so that if the classifier is used on new audio, we make sure to embed in a compatible way.

With this information, the classifier can be saved and used on other search sets later on.

## 8. Save your classifier for future use and use it for detection

In [ ]:

classifier_name = 'ybg_01'


classifier_path = config.models_folder / f'{classifier_name}.json'
classifier_path.parent.mkdir(exist_ok=True, parents=True)
agile.classifier.save(classifier_path)

We can also use the model to perform detection over all of the search dataset and save the results to a CSV file.

run the model over all of the search dataset and save the results to a csv. Specify:
1. `output_filename` The csv filename to save the results to
2. `threshold` The threshold. Anything above the threshold for the target labels will be saved. The classifier outputs a score for every embedding in the search set; anything above the threshold is considered a detection. A higher threshold value results in fewer but more confident detections, at the cost of missing out on relevant events in the search set. Conversely, a lower threshold value results in more but less confident detections, at the cost of producing more false positive detections. A threshold of zero is what the believes is the best threshold based on the training data, but this is often not perfect, so you may want to increase this to exclude more false positives or decrease it to include more false negatives (positive examples that got a negative score)
3. `subset` a number between 0 and 1 for the proportion of your dataset to classify. This is useful to check how things are going if you have a very large dataset.
4. `labels` Which labels to include. Leave it as None to include all the labels you trained your classifier for.




In [ ]:
output_filename = 'output.csv'

threshold = 1

# e.g. 0.5 for 50%.
subset = 0.001

# Which labels to include in the output file. If None, all labels are included.
labels = None

output_filepath = config.predictions_folder / output_filename
output_filepath.parent.mkdir(parents=True, exist_ok=True)
agile.run_inference(
    output_filepath,
    threshold=threshold,
    dataset=config.search_dataset_name,
    subset=subset,
)

# Verify the detections

Ecosounds provides a way for you to quickly scan through and mark these detections as true of false. 

Run the cell below to import the csv of detections you have just created into the ecosounds platform. It will generate a link where you can verify them



In [ ]:
project_id = None

agile.upload_detections(output_filepath, project_id)